In [ ]:
%load_ext autoreload
%autoreload 2

### 1. 데이터셋 생성

In [ ]:
# 모듈 임포트
from modules.sdf_generator import create_sample_shape, to_grid
import numpy as np

# 1. 도형 생성 (Phase 1)
print("--- Phase 1: SDF 그리드 생성 ---")
my_shape = create_sample_shape()

In [ ]:
import numpy as np
from modules.sdf_generator import create_random_shape, to_grid
from modules.particle_sampler import sample_particles_poisson  # 여기를 변경!
from modules.visualizer import visualize_simulation

# --- 설정 ---
NUM_SAMPLES = 5         # 테스트할 개수
RESOLUTION = 64         # 그리드 해상도
NUM_PARTICLES = 2000    # 목표 파티클 개수 (도형 내부를 채울 개수)

print(f"🚀 Poisson Disk Sampling 테스트 시작 ({NUM_SAMPLES}개)")

for i in range(NUM_SAMPLES):
    seed = 2024 + i  # 시드값 변경 (매번 다른 모양)
    
    # 1. 랜덤 도형 생성
    print(f"\n[{i+1}/{NUM_SAMPLES}] 도형 생성 중 (Seed: {seed})...")
    random_shape = create_random_shape(seed=seed)
    sdf_grid = to_grid(random_shape, resolution=RESOLUTION, domain_size=2.0)
    
    # 2. Poisson Disk Sampling 실행
    # (아까 수정한 로직 덕분에, 내부 부피를 계산해서 약 2000개를 맞춰줍니다)
    particles = sample_particles_poisson(
        sdf_grid, 
        domain_size=2.0, 
        num_particles=NUM_PARTICLES
    )
    
    print(f"   -> 생성된 파티클: {len(particles)}개")
    
    # 3. 시각화 (왼쪽: 단면, 오른쪽: 3D)
    visualize_simulation(
        sdf_grid=sdf_grid, 
        particles=particles, 
        domain_size=2.0, 
        title=f"Poisson Sample {i+1} (N={len(particles)})"
    )

print("\n✅ 테스트 완료!")

### 1-2 sdf -> 파티클 샘플링 

In [ ]:

# my shape -> sdf 해상도 64로 설정
resolution = 64
sdf_grid = to_grid(my_shape, resolution=resolution, domain_size=2.0)

# 입자 샘플링을 위해 Numpy 포맷으로 저장
np.save("sdf_grid_64.npy", sdf_grid)

print(f"Grid Shape: {sdf_grid.shape}")
print(f"Min Value: {sdf_grid.min():.3f}, Max Value: {sdf_grid.max():.3f}")

In [ ]:
import os
import numpy as np
from modules.sdf_generator import create_random_shape, to_grid
from modules.visualizer import visualize_simulation # 시각화용

# --- 설정값 ---
DATASET_SIZE = 5      # 생성할 데이터 개수
START_SEED = 42       # 시작 시드값 (재현 가능)
RESOLUTION = 64       # 그리드 해상도
OUTPUT_DIR = "dataset" # 저장할 폴더

# 폴더 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🚀 랜덤 데이터셋 생성 시작 (수량: {DATASET_SIZE}개)")

for i in range(DATASET_SIZE):
    # 1. 시드 설정 (각 데이터마다 시드가 달라야 함)
    current_seed = START_SEED + i
    
    # 2. 랜덤 도형 생성
    print(f"[{i+1}/{DATASET_SIZE}] Generating Shape (Seed: {current_seed})...")
    random_sdf = create_random_shape(seed=current_seed)
    
    # 3. 그리드로 변환
    sdf_grid = to_grid(random_sdf, resolution=RESOLUTION, domain_size=2.0)
    
    # 4. 저장 (NPY)
    # 파일명: shape_000.npy, shape_001.npy ...
    filename = os.path.join(OUTPUT_DIR, f"shape_{i:03d}.npy")
    np.save(filename, sdf_grid)
    
    print(f"\n👀 {i} 번째 데이터 미리보기:")
    visualize_simulation(sdf_grid=sdf_grid, title=f"Sample {i} (Seed {current_seed})")

print(f"\n✅ 생성 완료! '{OUTPUT_DIR}' 폴더를 확인하세요.")

### 2. sdf 계산 -CNN 학습

### 3. SDF 시각화

3.2 obj로 저장(mc 알고리즘)

In [ ]:
from modules.visualizer import save_mesh_as_obj # 시각화용
save_mesh_as_obj(sdf_grid, filename="test_shape_64.obj")


npy 파일 이용한 시각화

In [ ]:
from modules.visualizer import visualize_npy
# 2. 데이터 시각화 (Phase 2)
print("\n--- Phase 2: 시각화 및 저장 ---")

# SDF 파일 확인
visualize_npy("sdf_grid_64.npy")

# 파티클 파일 확인
visualize_npy("particles.npy")


변수(객체) 이용한 시각화(런타임에 있는 메모리)

In [ ]:
from modules.visualizer import visualize_simulation

# SDF만 보기 (기존 plot_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid)

# 파티클까지 겹쳐서 보기 (기존 plot_particles_and_sdf_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid, particles=particles)

In [ ]:
from modules.particle_sampler import sample_particles_poisson, sample_particles_jittered, sample_particles_dart_throwing
from modules.visualizer import visualize_simulation

# SDF 그리드 불러오기
sdf_grid = np.load(f"{OUTPUT_DIR}/shape_000.npy")

# Poisson Disk 방식 샘플링
particles = sample_particles_poisson(sdf_grid, domain_size=2.0, num_particles=2000)
print(f"샘플링된 파티클 수: {len(particles)}")

visualize_simulation(sdf_grid=sdf_grid, particles=particles, domain_size=2.0)
